# Quantum Cryptography: Factorisation & Lattice Problems on Hybrid Hardware

This notebook explores quantum algorithms for cryptographic problems on the
uniqx platform:

| Workload | Key op | Hardware |
|:---------|:-------|:---------|
| Period finding (Shor-like) | `expv` + `matmul` (QFT) | GPU / QPU |
| Lattice analysis (SVP) | `eigs` (Gram matrix) | CPU at low dim, GPU at high |
| Discrete logarithm | `expv` + `matmul` | GPU / QPU |

We compare hardware options across problem sizes, showing how the cost model
routes small factorisation problems to CPU and larger ones to GPU/QPU.

**All computation goes through uniqx.**

In [ ]:
import os
from uniqx import connect, login

endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("Connected to", endpoint)
import time
import matplotlib.pyplot as plt

from uniqx.domains.optimization.crypto import (
    build_period_finding_module,
    build_lattice_module,
    build_discrete_log_module,
    FACTORING_EXAMPLES,
)
from uniqx import parse_result
from uniqx.core.execution import connect, preflight, submit, get

API_KEY = os.environ.get("UNIQX_API_KEY")


## 1. Factoring Examples

In [ ]:
print(
    f"{'N':>5} {'Factors':>12} {'a':>4} {'Order':>6} {'Qubits':>7} {'Unitary dim':>12}"
)
print("-" * 48)
for N, info in FACTORING_EXAMPLES.items():
    print(
        f"{N:>5} {str(info['factors']):>12} {info['a']:>4} {info['order']:>6} {info['n_qubits']:>7} {N:>12}"
    )

## 2. Period Finding (Shor-like): Preflight Comparison

In [12]:
def print_opts(label, options):
    print(f"\n--- {label} ---")
    if not options:
        return
    for o in options:
        f = " *" if o.get("recommended") else ""
        print(
            f"  {o['label']:<20} time={o['total_time']:>8.1f}  cost=${o['total_cost_usd']:.4f}  err={o['max_error_rate']:.4f}{f}"
        )


period_results = {}

for N in [15, 21, 35]:
    info = FACTORING_EXAMPLES[N]
    mod, inputs, meta = build_period_finding_module(N=N, a=info["a"])
    opts = preflight(mod, client=client)
    print_opts(f"Factor N={N} (dim={meta['dim']})", opts)

    period_results[N] = {}
    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0

        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()
        out = parse_result(payload, ["transformed", "prob"])

        period_results[N][opt["label"]] = {"time": wall, "option": opt, "output": out}
        print(f"  {opt['label']}: {wall:.2f}s")

ValueError: status: Internal, message: "h2 protocol error: http2 error", details: [], metadata: MetadataMap { headers: {} }

## 3. Lattice Analysis: Cryptographic Hardness

In [14]:
lattice_results = {}

for dim in [4, 8, 12, 16]:
    mod, inputs, meta = build_lattice_module(dim=dim, q=101)
    opts = preflight(mod, client=client)
    print_opts(f"Lattice dim={dim} (Gram {dim}x{dim})", opts)

    lattice_results[dim] = {}
    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0

        payload = res.get("payload", b"")
        if isinstance(payload, str):
            payload = payload.encode()
        out = parse_result(payload, ["eigenvalues", "eigenvectors"])
        evals = out.get("eigenvalues", ([], "", []))[2] if out else []

        lattice_results[dim][opt["label"]] = {
            "time": wall,
            "eigenvalues": evals,
            "option": opt,
        }
        if evals:
            ratio = max(abs(e) for e in evals) / max(
                min(abs(e) for e in evals if abs(e) > 1e-10), 1e-10
            )
            print(f"  {opt['label']}: {wall:.2f}s, λ_max/λ_min = {ratio:.1f}")
        else:
            print(f"  {opt['label']}: {wall:.2f}s")

ValueError: status: Internal, message: "h2 protocol error: http2 error", details: [], metadata: MetadataMap { headers: {} }

## 4. Discrete Logarithm

In [15]:
dlog_results = {}

for g, p in [(2, 13), (3, 17), (2, 23)]:
    mod, inputs, meta = build_discrete_log_module(g=g, p=p)
    opts = preflight(mod, client=client)
    print_opts(f"DLog g={g}, p={p} (dim={meta['dim']})", opts)

    dlog_results[(g, p)] = {}
    for opt in opts:
        t0 = time.monotonic()
        jid = submit(
            mod,
            client=client,
            runtime_inputs=inputs,
            preflight_job_id=opts.job_id,
            option_idx=opt["_idx"],
        )
        res = get(jid, client=client)
        wall = time.monotonic() - t0

        dlog_results[(g, p)][opt["label"]] = {"time": wall, "option": opt}
        print(f"  {opt['label']}: {wall:.2f}s")

{"timestamp": "2026-05-21T20:27:20+0200", "level": "WARNING", "component": "sdk", "subcomponent": "execution", "file": "/Users/alessandro/Documents/hackathon_21-22_05_26/venv/lib/python3.14/site-packages/uniqx/core/execution.py", "line": 78, "message": "transient gateway error on submit (attempt 1/3): status: Unavailable, message: \"unavailable\", details: [], metadata: MetadataMap { headers: {\"server\": \"awselb/2.0\", \"date\": \"Thu, 21 May 2026 18:27:20 GMT\", \"content-type\": \"application/grpc\", \"content-length\": \"0\"} }", "job_id": "0ea9f830-7dea-4c62-b1c0-d67b7d452c9d"}
{"timestamp": "2026-05-21T20:27:45+0200", "level": "WARNING", "component": "sdk", "subcomponent": "execution", "file": "/Users/alessandro/Documents/hackathon_21-22_05_26/venv/lib/python3.14/site-packages/uniqx/core/execution.py", "line": 78, "message": "transient gateway error on submit (attempt 2/3): status: Unavailable, message: \"unavailable\", details: [], metadata: MetadataMap { headers: {\"server\": 

UniqxNetworkError: Connection to server lost. Check your network and try again.

## 5. Scaling Analysis

In [ ]:
# Period-finding scaling
pf_scaling = []
for N in [15, 21, 33, 35, 55]:
    info = FACTORING_EXAMPLES[N]
    mod, inputs, meta = build_period_finding_module(N=N, a=info["a"])
    opts = preflight(mod, client=client)
    print(f"\nN={N} (dim={meta['dim']}):")
    for opt in opts:
        f = " *" if opt.get("recommended") else ""
        print(
            f"  {opt['label']:<20} time={opt['total_time']:>8.1f}  cost=${opt['total_cost_usd']:.4f}{f}"
        )
        pf_scaling.append(
            {
                "N": N,
                "dim": meta["dim"],
                "label": opt["label"],
                "time": opt["total_time"],
                "cost": opt["total_cost_usd"],
                "recommended": opt.get("recommended", False),
            }
        )

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors_hw = {"cpu-only": "#2563eb", "cpu+gpu": "#16a34a", "cpu+gpu+qpu": "#9333ea"}

# Top-left: Period-finding execution time by N
Ns = sorted(period_results.keys())
for hw in sorted(set(l for n in period_results for l in period_results[n])):
    times = [period_results[n].get(hw, {}).get("time", 0) for n in Ns]
    c = colors_hw.get(hw, "#6b7280")
    axes[0, 0].plot(Ns, times, "o-", color=c, label=hw)
axes[0, 0].set_xlabel("N (number to factor)")
axes[0, 0].set_ylabel("Wall time (s)")
axes[0, 0].set_title("Period Finding: Execution Time")
axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(alpha=0.3)

# Top-right: Lattice eigenvalue spectra
for dim_val in sorted(lattice_results.keys()):
    # Use first available hardware result
    for label, data in lattice_results[dim_val].items():
        if data["eigenvalues"]:
            evals = sorted(data["eigenvalues"])
            axes[0, 1].semilogy(
                range(len(evals)),
                [abs(e) for e in evals],
                "o-",
                label=f"dim={dim_val}",
                markersize=3,
            )
            break
axes[0, 1].set_xlabel("Eigenvalue index")
axes[0, 1].set_ylabel("|λ| (log scale)")
axes[0, 1].set_title("Lattice Gram Matrix Spectrum")
axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(alpha=0.3)

# Bottom-left: Discrete log time by prime size
primes = sorted(dlog_results.keys(), key=lambda x: x[1])
for hw in sorted(set(l for k in dlog_results for l in dlog_results[k])):
    ps = [k[1] for k in primes]
    times = [dlog_results[k].get(hw, {}).get("time", 0) for k in primes]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 0].plot(ps, times, "o-", color=c, label=hw)
axes[1, 0].set_xlabel("Prime p")
axes[1, 0].set_ylabel("Wall time (s)")
axes[1, 0].set_title("Discrete Logarithm: Scaling")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

# Bottom-right: Preflight scaling for period finding
hw_labels = sorted(set(d["label"] for d in pf_scaling))
for hw in hw_labels:
    sub = [d for d in pf_scaling if d["label"] == hw]
    c = colors_hw.get(hw, "#6b7280")
    axes[1, 1].semilogy(
        [d["dim"] for d in sub], [d["time"] for d in sub], "o-", color=c, label=hw
    )
axes[1, 1].set_xlabel("Unitary dimension")
axes[1, 1].set_ylabel("Estimated time (log)")
axes[1, 1].set_title("Period Finding: Hardware Scaling")
axes[1, 1].legend(fontsize=8)
axes[1, 1].grid(alpha=0.3)

fig.suptitle("Quantum Cryptography on Hybrid Hardware", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Summary

| Problem | Dimension | Best hardware | Key insight |
|:--------|:----------|:--------------|:------------|
| Factor N=15 | dim=15 | CPU | Small unitary, no transfer overhead |
| Factor N=35 | dim=35 | GPU | expv+matmul benefit from parallelism |
| Factor N=55+ | dim=55 | GPU / QPU | QPU advantage for quantum simulation |
| Lattice dim=4 | 4x4 Gram | CPU | Tiny eigendecomposition |
| Lattice dim=16 | 16x16 Gram | GPU crossover | Dense eigs scales quadratically |
| Discrete log p=23 | dim=23 | GPU | Same expv+QFT structure as Shor |

uniqx routes small quantum simulations to CPU and larger ones to GPU/QPU.
Lattice problems (eigs) follow the same crossover pattern as other eigendecompositions.
Period-finding and discrete log share the same expv+matmul structure, so hardware
preferences track unitary dimension.